In [1]:
import sys, os
from pathlib import Path

IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

if IS_KAGGLE:
    # Install packages not available on Kaggle
    %pip install -q kymatio kornia
    
    # Add repo to path (UPDATE 'deep-learning-course-project' to your dataset slug)
    repo_path = Path('/kaggle/input/deep-learning-course-project')
    if repo_path.exists():
        sys.path.insert(0, str(repo_path))
else:
    # Local: add project root to path (assumes notebook is in notebooks/)
    project_root = Path.cwd().parent
    if (project_root / 'src').exists():
        sys.path.insert(0, str(project_root))

from src.utils.config import *
from src.utils.datasets import get_cifar10_loaders_and_splits
from src.utils.training import calculate_accuracy, load_weights
from src.models.architectures.RestNet18 import *
from src.models.architectures.ScatNet18 import *
from src.models.architectures.AdditiveHybridGaborResNet18 import *

from kornia import augmentation as K

====================== Hyperparameters =======================
N_EPOCHS: 200
T_MAX: 200
CRITERION: CrossEntropyLoss()
DEVICE: cuda
SEED: 42
BATCH_SIZE: 128
LR: 0.001
MOMENTUM: 0.9
WEIGHT_DECAY: 0.0001
Setting seed to 42


In [2]:
DEBUG = True
EXP_NAME = "models_robustness"
print(f"Starting experiment {EXP_NAME}. DEBUG={DEBUG}")

n_samples_per_class_train = [10, 50, 100, 500, 1000, 4000]

Starting experiment models_robustness. DEBUG=True


In [3]:
# Get Augmentations
augmentations = {
    'Horizontal Flip': K.RandomHorizontalFlip(p=1.0, same_on_batch=True, keepdim=True).to(DEVICE),
    'Vertical Flip': K.RandomVerticalFlip(p=1.0, same_on_batch=True, keepdim=True).to(DEVICE),
    'Gray Scale': K.RandomGrayscale(same_on_batch=True, p=1.0, keepdim=True).to(DEVICE),
    'Gaussian Noise': K.RandomGaussianNoise(mean=0.0, std=0.05, same_on_batch=True, keepdim=True, p=1.0).to(DEVICE),
    'Gaussian Blur': K.RandomGaussianBlur(kernel_size=(3,3), sigma=(2.0,2.0), p=1.0, same_on_batch=True, keepdim=True).to(DEVICE),
    'Median Blur': K.RandomMedianBlur(kernel_size=(3,3), p=1.0, same_on_batch=True, keepdim=True).to(DEVICE)
}

In [ ]:
# Get models
resnet_models = {}
scatnet_models = {}
max_hybridgabor_layer2_models = {}
max_hybridgabor_layer3_models = {}

for n_samples in n_samples_per_class_train:
    resnet_models[n_samples] = MakeResNet18().to(DEVICE)
    load_weights(
        resnet_models[n_samples],
        experiment_name="baseline_acc_vs_n_samples",
        model_name=f"ResNet18_{n_samples}",
        device=DEVICE,
        kaggle_load=False
    )

    scatnet_models[n_samples] = MakeScatResNet18(L=10).to(DEVICE)
    load_weights(
        scatnet_models[n_samples],
        experiment_name="scatresnet_acc_vs_n_samples",
        model_name=f"ScatResNet18_{n_samples}",
        device=DEVICE,
        kaggle_load=False
    )

    max_hybridgabor_layer2_models[n_samples] = MakeAdditiveHybridGaborResNet18(L=2).to(DEVICE)
    load_weights(
        max_hybridgabor_layer2_models[n_samples],
        experiment_name="maxhybridgabor_acc_layers_vs_n_samples",
        model_name=f"AdditiveHybridGaborResNet18_L2_{n_samples}",
        device=DEVICE,
        kaggle_load=False,
        strict=False
    )

    max_hybridgabor_layer3_models[n_samples] = MakeAdditiveHybridGaborResNet18(L=3).to(DEVICE)
    load_weights(
        max_hybridgabor_layer3_models[n_samples],
        experiment_name="maxhybridgabor_acc_layers_vs_n_samples",
        model_name=f"AdditiveHybridGaborResNet18_L3_{n_samples}",
        device=DEVICE,
        kaggle_load=False,
        strict=False
    )

10
L2
L3
50
L2
L3
100
L2
L3
500
L2
L3
1000
L2
L3
4000
L2
L3


In [ ]:
# Get test loader
_, _, testloader, _, _, test_set = get_cifar10_loaders_and_splits()

In [ ]:
# Visualize augmentations
if not DEBUG:
    img, label = test_set[0]

    # img is (C, H, W) tensor, needs to be (H, W, C) for matplotlib
    img_np = img.permute(1, 2, 0).numpy()

    fig, axes = plt.subplots(1, len(augmentations) + 1, figsize=(3 * (len(augmentations) + 1), 3))

    # Show original
    axes[0].imshow(img_np)
    axes[0].set_title("Original")
    axes[0].axis('off')

    # Show each augmentation
    for i, (aug_name, aug) in enumerate(augmentations.items()):
        # Add batch dimension, apply aug, remove batch dimension
        aug_img = aug(img.unsqueeze(0).to(DEVICE)).squeeze(0).cpu()
        aug_img_np = aug_img.permute(1, 2, 0).numpy()
        
        # Clip values to [0, 1] for display (noise can push values outside)
        aug_img_np = aug_img_np.clip(0, 1)
        
        axes[i + 1].imshow(aug_img_np)
        axes[i + 1].set_title(aug_name)
        axes[i + 1].axis('off')

    plt.suptitle(f"Test Image (Label: {label})", fontsize=12)
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / "augmentations_visualization.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Check models accuracy on each augmentation
resnet_accs = {}
scatnet_accs = {}
max_hybridgabor_layer2_accs = {}
max_hybridgabor_layer3_accs = {}

if not DEBUG:
    # Resnet
    for n_samples in n_samples_per_class_train:
        resnet_accs[n_samples] = {}
        model = resnet_models[n_samples]
        for aug_name, aug in augmentations.items():
            print(f"Checking accuracy for Resnet_{n_samples} on augmentation {aug}")
            acc = calculate_accuracy(
                model, testloader, DEVICE, augmentations=aug
            )
            resnet_accs[n_samples][aug_name] = acc
            print(f"Accuracy: {acc}")

    # ScatNet
    for n_samples in n_samples_per_class_train:
        scatnet_accs[n_samples] = {}
        model = scatnet_models[n_samples]
        for aug_name, aug in augmentations.items():
            print(f"Checking accuracy for ScatResnet_{n_samples} on augmentation {aug}")
            acc = calculate_accuracy(
                model, testloader, DEVICE, augmentations=aug
            )
            scatnet_accs[n_samples][aug_name] = acc
            print(f"Accuracy: {acc}")

    # Max HybridGabor Layer 2
    for n_samples in n_samples_per_class_train:
        max_hybridgabor_layer2_accs[n_samples] = {}
        model = max_hybridgabor_layer2_models[n_samples]
        for aug_name, aug in augmentations.items():
            print(f"Checking accuracy for MaxHybridGaborLayer2_{n_samples} on augmentation {aug}")
            acc = calculate_accuracy(
                model, testloader, DEVICE, augmentations=aug
            )
            max_hybridgabor_layer2_accs[n_samples][aug_name] = acc
            print(f"Accuracy: {acc}")

    # Max HybridGabor Layer 3
    for n_samples in n_samples_per_class_train:
        max_hybridgabor_layer3_accs[n_samples] = {}
        model = max_hybridgabor_layer3_models[n_samples]
        for aug_name, aug in augmentations.items():
            print(f"Checking accuracy for MaxHybridGaborLayer3_{n_samples} on augmentation {aug}")
            acc = calculate_accuracy(
                model, testloader, DEVICE, augmentations=aug
            )
            max_hybridgabor_layer3_accs[n_samples][aug_name] = acc
            print(f"Accuracy: {acc}")


In [ ]:
# Plot results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

aug_names = list(augmentations.keys())
n_samples_list = list(n_samples_per_class_train)

for i, aug_name in enumerate(aug_names):
    ax = axes[i]
    
    # Get accuracies for each n_samples
    resnet_acc_list = [resnet_accs[n][aug_name] for n in n_samples_list]
    scatnet_acc_list = [scatnet_accs[n][aug_name] for n in n_samples_list]
    max_hybridgabor_layer2_acc_list = [max_hybridgabor_layer2_accs[n][aug_name] for n in n_samples_list]
    max_hybridgabor_layer3_acc_list = [max_hybridgabor_layer3_accs[n][aug_name] for n in n_samples_list]

    # Plot
    ax.plot(n_samples_list, resnet_acc_list, 'o-', label='ResNet18', color='blue', markersize=8)
    ax.plot(n_samples_list, scatnet_acc_list, 's-', label='ScatNet18', color='orange', markersize=8)
    ax.plot(n_samples_list, max_hybridgabor_layer2_acc_list, '^-', label='MaxHybridGaborLayer2', color='green', markersize=8)
    ax.plot(n_samples_list, max_hybridgabor_layer3_acc_list, 'x-', label='MaxHybridGaborLayer3', color='red', markersize=8)
    
    ax.set_xlabel('Samples per Class (Log)')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title(f'{aug_name}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log')

# Hide unused subplots if any
for j in range(len(aug_names), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Model Robustness: ResNet18 vs ScatNet18', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_PATH / f"{EXP_NAME}.png", dpi=150, bbox_inches='tight')
plt.show()

Robusteness of Models - Differences between accuracy on augmented and un-augmented images

In [ ]:
# Calculate accuracy of models on un-augmented images
un_aug_resnet_accs = {}
un_aug_scatnet_accs = {}
un_aug_max_hybridgabor_layer2_accs = {}
un_aug_max_hybridgabor_layer3_accs = {}

if not DEBUG:
    # Resnet
    for n_samples in n_samples_per_class_train:
        model = resnet_models[n_samples]
        print(f"Checking accuracy for Resnet_{n_samples} on un-augmented images")
        acc = calculate_accuracy(
            model, testloader, DEVICE,
        )
        un_aug_resnet_accs[n_samples] = acc
        print(f"Accuracy: {acc}")

    # ScatNet
    for n_samples in n_samples_per_class_train:
        model = scatnet_models[n_samples]
        print(f"Checking accuracy for ScatResnet_{n_samples} on un-augmented images")
        acc = calculate_accuracy(
            model, testloader, DEVICE,
        )
        un_aug_scatnet_accs[n_samples] = acc
        print(f"Accuracy: {acc}")

    # Max HybridGabor Layer 2
    for n_samples in n_samples_per_class_train:
        model = max_hybridgabor_layer2_models[n_samples]
        print(f"Checking accuracy for MaxHybridGaborLayer2_{n_samples} on un-augmented images")
        acc = calculate_accuracy(
            model, testloader, DEVICE
        )
        un_aug_max_hybridgabor_layer2_accs[n_samples] = acc
        print(f"Accuracy: {acc}")

    # Max HybridGabor Layer 3
    for n_samples in n_samples_per_class_train:
        model = max_hybridgabor_layer3_models[n_samples]
        print(f"Checking accuracy for MaxHybridGaborLayer3_{n_samples} on un-augmented images")
        acc = calculate_accuracy(
            model, testloader, DEVICE
        )
        un_aug_max_hybridgabor_layer3_accs[n_samples] = acc
        print(f"Accuracy: {acc}")


In [ ]:
# Plot results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

aug_names = list(augmentations.keys())
n_samples_list = list(n_samples_per_class_train)

for i, aug_name in enumerate(aug_names):
    ax = axes[i]
    
    # Get accuracies for each n_samples
    resnet_acc_list = [resnet_accs[n][aug_name] - un_aug_resnet_accs[n] for n in n_samples_list]
    scatnet_acc_list = [scatnet_accs[n][aug_name] - un_aug_scatnet_accs[n] for n in n_samples_list]
    max_hybridgabor_layer2_acc_list = [max_hybridgabor_layer2_accs[n][aug_name] - un_aug_max_hybridgabor_layer2_accs[n] for n in n_samples_list]
    max_hybridgabor_layer3_acc_list = [max_hybridgabor_layer3_accs[n][aug_name] - un_aug_max_hybridgabor_layer3_accs[n] for n in n_samples_list]
    
    # Plot
    ax.plot(n_samples_list, resnet_acc_list, 'o-', label='ResNet18', color='blue', markersize=8)
    ax.plot(n_samples_list, scatnet_acc_list, 's-', label='ScatNet18', color='orange', markersize=8)
    ax.plot(n_samples_list, max_hybridgabor_layer2_acc_list, '^-', label='MaxHybridGaborLayer2', color='green', markersize=8)
    ax.plot(n_samples_list, max_hybridgabor_layer3_acc_list, 'x-', label='MaxHybridGaborLayer3', color='red', markersize=8)
    
    ax.set_xlabel('Samples per Class (Log)')
    ax.set_ylabel('Accuracy Drop (%)')
    ax.set_title(f'{aug_name}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log')

# Hide unused subplots if any
for j in range(len(aug_names), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Models Robustness on Augmentations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_PATH / f"{EXP_NAME}.png", dpi=150, bbox_inches='tight')
plt.show()